# 03 · Gold — Analitica, agregados y features

| | |
|---|---|
| **Objetivo** | Preparar una tabla por consumidor, con el contrato que cada uno necesita |
| **Entradas** | `silver.clean_trips` |
| **Salidas** | `gold.trips_analytics`, `gold.agg_demanda_zona_hora`, `gold.trips_features` |
| **Depende de** | Notebook 02 |

**Proceso**
1. Derivar features temporales, geograficas y la bandera de aeropuerto
2. Asignar el split temporal como columna materializada
3. Ajustar el agrupamiento de zonas solo sobre la particion de entrenamiento
4. Construir `trips_analytics` con el detalle completo, particionada por mes
5. Construir `agg_demanda_zona_hora` con el grano zona x hora x dia
6. Construir `trips_features` por lista blanca, particionada por split
7. Verificar que ninguna columna prohibida sobrevivio

**Decision de diseno.** Gold se divide segun quien consume cada tabla, y
esa separacion es la barrera contra fugas de datos:

- `trips_analytics` conserva todo, incluida la velocidad y la hora de
  llegada. Alimenta el analisis exploratorio.
- `agg_demanda_zona_hora` precalcula los agregados que consumen las
  visualizaciones, evitando recorrer el detalle en cada consulta.
- `trips_features` contiene unicamente lo conocido en el instante de la
  recogida. Alimenta el modelo.

La velocidad promedio se calcula como distancia sobre duracion, de modo que
deriva del target: es legitima para describir el fenomeno e invalida como
variable de entrada. Separar las tablas hace que la arquitectura garantice
esa distincion en lugar de depender de recordarla.

In [0]:
import os
import sys


# Localiza la raiz del repo subiendo hasta encontrar src/, en vez de fijar un
# numero de saltos. Asi el notebook funciona a cualquier profundidad.
def _preparar_path(marcador="src", max_niveles=10):
    ruta = os.getcwd()
    for _ in range(max_niveles):
        if os.path.isdir(os.path.join(ruta, marcador)):
            destino = os.path.join(ruta, marcador)
            if destino not in sys.path:
                sys.path.insert(0, destino)
            return destino
        padre = os.path.dirname(ruta)
        if padre == ruta:
            break
        ruta = padre
    raise RuntimeError(f"No se encontro la raiz del repo (carpeta con {marcador}/)")


_preparar_path()

from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import functions as F

from nyc_taxi import config
from nyc_taxi.data_prep import splits
from nyc_taxi.features import transformations as tf

In [0]:
df = spark.table(config.TBL_CLEAN)
n_silver = df.count()
print(f"Filas en Silver: {n_silver:,}")

Filas en Silver: 1,455,526


## 1. Features derivadas

Todo el calculo geografico usa funciones nativas de Spark, sin UDFs de
Python. Un UDF resultaria mas legible pero obliga a serializar cada fila
hacia un interprete de Python y de vuelta, lo que sobre 1,45 millones de
registros cuesta varias veces mas. Las definiciones equivalentes en Python
puro viven en `features/definitions.py` y estan cubiertas por pruebas.

Se calculan tres medidas de trayecto porque capturan cosas distintas: la
haversine mide la linea recta, la de cuadricula aproxima el recorrido real
sobre la retícula de calles de la ciudad, y el rumbo captura la direccion,
que no es simetrica: entrar a Manhattan y salir de ella tienen perfiles de
congestion distintos aunque la distancia sea identica.

In [0]:
df = tf.agregar_features_temporales(df)
df = tf.agregar_features_geograficas(df)
df = df.withColumn(
    "es_aeropuerto",
    tf.col_es_aeropuerto(
        config.AEROPUERTOS, config.RADIO_AEROPUERTO_KM,
        config.KM_POR_GRADO_LAT, config.KM_POR_GRADO_LON_NYC,
    ),
)

## 2. Split temporal materializado

El reparto se guarda como columna en lugar de recalcularse en cada corrida
de entrenamiento. Asi cualquier notebook que lea la tabla obtiene la misma
particion, y el criterio queda auditable en el propio dato.

El corte es temporal y no aleatorio porque el uso real del modelo es
estimar viajes futuros. Un reparto aleatorio dejaria trayectos del mismo
dia, la misma hora y la misma congestion a ambos lados, de modo que la
metrica mediria memorizacion de condiciones concretas y no generalizacion.

In [0]:
df = df.withColumn("split_flag", splits.col_split())

reparto = df.groupBy("split_flag").count().orderBy("split_flag").collect()
for fila in reparto:
    print(f"{fila['split_flag']:<6} {fila['count']:>10,}  ({fila['count'] / n_silver * 100:.1f}%)")

test      106,391  (7.3%)
train   1,349,135  (92.7%)


## 3. Materializacion intermedia

El computo serverless no admite `cache()`. Sin persistir de alguna forma,
el DataFrame se recomputaria completo en cada escritura posterior. Escribir
una tabla Delta intermedia cumple el mismo proposito: se evalua una vez y
las lecturas siguientes no vuelven a calcular las expresiones geograficas.

In [0]:
TBL_ENRIQUECIDO = f"{config.CATALOGO}.{config.SCHEMA_GOLD}._trips_enriquecido"

(df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(TBL_ENRIQUECIDO))

df = spark.table(TBL_ENRIQUECIDO)

## 4. Agrupamiento de zonas

Las cuatro coordenadas crudas son poco informativas para un modelo de
arboles, que tendria que aprender los limites de cada barrio a base de
cortes sucesivos. Agruparlas en zonas convierte esa geometria en dos
variables categoricas que el modelo aprovecha directamente.

**El modelo se ajusta unicamente sobre la particion de entrenamiento.**
Ajustarlo sobre el conjunto completo definiria los centroides usando
tambien los viajes de evaluacion, lo que introduce informacion del futuro
en las variables de entrada. Es una fuga sutil, porque no involucra al
target, y por eso mismo es facil de pasar por alto.

Se ajusta un solo modelo sobre las coordenadas de recogida y se aplica a
ambos extremos, de modo que una zona signifique lo mismo como origen que
como destino.

In [0]:
ensamblador_pickup = VectorAssembler(
    inputCols=["pickup_latitude", "pickup_longitude"], outputCol="coords"
)
ensamblador_dropoff = VectorAssembler(
    inputCols=["dropoff_latitude", "dropoff_longitude"], outputCol="coords"
)

coords_entrenamiento = ensamblador_pickup.transform(
    df.filter(F.col("split_flag") == splits.SPLIT_TRAIN)
).select("coords")

modelo_zonas = KMeans(
    featuresCol="coords",
    predictionCol="zona",
    k=config.N_CLUSTERS_ZONA,
    seed=config.SEMILLA,
).fit(coords_entrenamiento)

print(f"Zonas ajustadas sobre {coords_entrenamiento.count():,} viajes de entrenamiento")

Zonas ajustadas sobre 1,349,135 viajes de entrenamiento


In [0]:
df_zonas = (
    modelo_zonas.transform(ensamblador_pickup.transform(df))
    .withColumnRenamed("zona", "pickup_cluster")
    .drop("coords")
)
df_zonas = (
    modelo_zonas.transform(ensamblador_dropoff.transform(df_zonas))
    .withColumnRenamed("zona", "dropoff_cluster")
    .drop("coords")
)

TBL_ZONAS = f"{config.CATALOGO}.{config.SCHEMA_GOLD}._trips_zonas"

(df_zonas.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(TBL_ZONAS))

df_zonas = spark.table(TBL_ZONAS)

## 5. `trips_analytics`

Grano de viaje, con todo el detalle. Es la unica tabla donde aparece la
velocidad promedio, precisamente porque no puede llegar al modelo.
Incluye tambien las zonas, que el analisis exploratorio necesita para
estudiar los pares origen-destino.

Particionada por mes: seis particiones de unas 240 mil filas. A este
volumen el particionamiento es demostrativo mas que necesario, y conviene
decirlo: el criterio real en produccion seria un tamano de particion
objetivo de entre 128 MB y 1 GB. Particionar por hora del dia generaria 24
carpetas de archivos diminutos y degradaria el rendimiento en lugar de
mejorarlo, que es el error mas comun al aplicar este patron.

In [0]:
df_analytics = df_zonas.withColumn(
    "velocidad_kmh",
    F.when(F.col("trip_duration") > 0,
           F.col("distancia_haversine_km") / (F.col("trip_duration") / 3600.0))
    .otherwise(F.lit(0.0)),
).withColumn("duracion_min", F.col("trip_duration") / 60.0)

(df_analytics.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .partitionBy("pickup_month")
 .saveAsTable(config.TBL_ANALYTICS))

print(f"{config.TBL_ANALYTICS}: {spark.table(config.TBL_ANALYTICS).count():,} filas")

nyc_taxi.gold.trips_analytics: 1,455,526 filas


## 6. `agg_demanda_zona_hora`

Tabla de hechos agregada con grano de zona de origen por hora por dia de
la semana. Existe para que las visualizaciones del analisis exploratorio no
recorran 1,45 millones de filas en cada consulta: son unos pocos miles de
registros que responden lo mismo.

Se usa la mediana y no el promedio porque la distribucion de duraciones
esta fuertemente sesgada a la derecha, y unos pocos trayectos largos
desplazarian la media hacia arriba sin representar al viaje tipico.

In [0]:
df_agg = (
    df_zonas.groupBy("pickup_cluster", "pickup_hour", "pickup_dayofweek")
    .agg(
        F.count(F.lit(1)).alias("n_viajes"),
        F.expr("percentile_approx(trip_duration, 0.5)").alias("duracion_mediana_seg"),
        F.expr("percentile_approx(distancia_haversine_km, 0.5)").alias("distancia_mediana_km"),
        F.round(F.avg("passenger_count"), 2).alias("pasajeros_promedio"),
    )
    .withColumn(
        "velocidad_mediana_kmh",
        F.round(F.col("distancia_mediana_km") / (F.col("duracion_mediana_seg") / 3600.0), 2),
    )
    .withColumn("duracion_mediana_min", F.round(F.col("duracion_mediana_seg") / 60.0, 2))
)

(df_agg.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(config.TBL_AGG_DEMANDA))

print(f"{config.TBL_AGG_DEMANDA}: {spark.table(config.TBL_AGG_DEMANDA).count():,} filas")

nyc_taxi.gold.agg_demanda_zona_hora: 3,331 filas


## 7. `trips_features`

Seleccion por lista blanca, no por descarte. Con `drop()` se olvida una
columna y nadie se entera; con lista blanca, todo lo que no este declarado
en `config.FEATURES_MODELO` queda fuera por defecto.

El target se guarda en sus dos formas. La transformacion logaritmica
responde al sesgo de la distribucion: sin ella, el error cuadratico queda
dominado por los trayectos mas largos y el modelo optimiza el caso raro a
costa del habitual. Ademas alinea el entrenamiento con la metrica oficial
de la competencia, que es logaritmica.

Particionada por `split_flag`: cada corrida de entrenamiento lee su
particion sin recorrer la tabla completa. Particionar por mes no aportaria
nada aqui, porque el modelado nunca consulta por mes.

In [0]:
df_features = (
    df_zonas
    .withColumn(config.TARGET_LOG, F.log1p(F.col(config.TARGET)))
    .select(
        "id",
        *config.FEATURES_MODELO,
        config.TARGET,
        config.TARGET_LOG,
        "split_flag",
    )
)

(df_features.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .partitionBy("split_flag")
 .saveAsTable(config.TBL_FEATURES))

print(f"{config.TBL_FEATURES}: {spark.table(config.TBL_FEATURES).count():,} filas")

nyc_taxi.gold.trips_features: 1,455,526 filas


## 8. Verificacion anti fuga

Comprobacion explicita de que ninguna columna prohibida sobrevivio hasta la
tabla que alimenta el modelo. Convierte una intencion de diseno en una
garantia que el pipeline verifica en cada ejecucion.

In [0]:
columnas_features = set(spark.table(config.TBL_FEATURES).columns)
fugas = columnas_features & set(config.COLUMNAS_PROHIBIDAS) - {config.TARGET, config.TARGET_LOG}

assert not fugas, f"Columnas con fuga de datos en la tabla de features: {fugas}"

for prohibida in ["dropoff_datetime", "velocidad_kmh", "duracion_min"]:
    assert prohibida not in columnas_features, f"'{prohibida}' llego a la tabla de features"

print("Sin fugas. Columnas de la tabla de features:")
for c in sorted(columnas_features):
    print(f"  {c}")

Sin fugas. Columnas de la tabla de features:
  bearing_deg
  distancia_haversine_km
  distancia_manhattan_km
  dropoff_cluster
  es_aeropuerto
  id
  is_weekend
  log_trip_duration
  passenger_count
  pickup_cluster
  pickup_dayofweek
  pickup_hour
  pickup_month
  split_flag
  store_and_fwd_flag
  trip_duration
  vendor_id


## 9. Verificacion de integridad y limpieza

In [0]:
n_features = spark.table(config.TBL_FEATURES).count()
n_analytics = spark.table(config.TBL_ANALYTICS).count()

assert n_features == n_silver, f"trips_features perdio filas: {n_features:,} vs {n_silver:,}"
assert n_analytics == n_silver, f"trips_analytics perdio filas: {n_analytics:,} vs {n_silver:,}"

nulos = spark.table(config.TBL_FEATURES).select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in config.FEATURES_MODELO]
).first().asDict()
con_nulos = {c: n for c, n in nulos.items() if n > 0}
assert not con_nulos, f"Features con valores nulos: {con_nulos}"

spark.sql(f"DROP TABLE IF EXISTS {TBL_ENRIQUECIDO}")
spark.sql(f"DROP TABLE IF EXISTS {TBL_ZONAS}")

print("Capa Gold construida y verificada.")

Capa Gold construida y verificada.


In [0]:
display(spark.table(config.TBL_FEATURES).limit(10))

id,vendor_id,passenger_count,store_and_fwd_flag,pickup_hour,pickup_dayofweek,pickup_month,is_weekend,distancia_haversine_km,distancia_manhattan_km,bearing_deg,pickup_cluster,dropoff_cluster,es_aeropuerto,trip_duration,log_trip_duration,split_flag
id0161358,1,3,0,19,7,5,1,1.301625729791064,1.6239077446889212,16.903636983507113,0,0,0,378,5.937536205082426,train
id0327031,1,2,0,4,1,4,1,1.5312170536377838,2.1655322857710617,44.78355559842953,10,14,0,256,5.54907608489522,train
id0295462,1,1,0,21,3,2,0,2.062744660223025,2.433175101442187,11.517596201633637,5,8,0,463,6.139884552226255,train
id3961200,1,1,0,19,3,3,0,1.7701612301554048,2.4053345528194585,118.9124241947643,4,11,0,468,6.150602768446279,train
id1638464,2,1,0,12,4,1,0,1.5424727398159257,2.0906242955062546,298.413131659015,11,4,0,527,6.269096283706261,train
id0780329,2,1,0,10,4,5,0,2.2226820912307557,3.095319366726299,145.00724335833968,9,11,0,1129,7.029972911706386,train
id2293342,2,1,0,16,1,5,1,2.8609141073009146,3.4857416720378067,194.49643620005148,8,10,0,700,6.55250788703459,train
id1029504,2,2,0,1,5,2,0,1.8724187697507573,2.5498188104377695,299.347137157057,0,8,0,280,5.638354669333745,train
id1359840,1,1,0,18,6,4,0,3.3525281350757945,4.417096392532078,23.67784469064287,0,18,0,682,6.52649485957079,train
id1787828,1,1,0,17,4,3,0,1.1535779998115916,1.6022453774804972,235.84518407651956,4,4,0,315,5.755742213586912,train
